# 🛡️ AI Merchant Risk Analyzer — Experimentation Notebook

This notebook walks through the complete pipeline interactively:
1. Data Generation
2. EDA & Seasonality Analysis
3. Feature Engineering
4. NLP Risk Analysis
5. Anomaly Detection
6. Clustering (Elbow + Silhouette)
7. Risk Scoring


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
%matplotlib inline
print('Imports OK')

## 1. Generate Dataset

In [ ]:
from data_generation import generate_dataset, save_dataset
os.makedirs('../data', exist_ok=True)
df = generate_dataset(100_000)
save_dataset(df, '../data/transactions.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nCluster distribution:')
print(df['_cluster_label'].value_counts())

## 2. EDA — Seasonality & Festival Analysis

In [ ]:
df['Month'] = pd.to_datetime(df['Timestamp']).dt.month

# Monthly transaction volume
monthly = df.groupby('Month')['TotalVolume'].mean()
festival_months = [1, 3, 4, 8, 10, 12]

fig, ax = plt.subplots(figsize=(12,5))
colors = ['#e74c3c' if m in festival_months else '#3498db' for m in monthly.index]
ax.bar(monthly.index, monthly.values, color=colors)
ax.set_xlabel('Month')
ax.set_ylabel('Avg Total Volume (₹)')
ax.set_title('Average Transaction Volume by Month\n(Red = Festival Months)')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

In [ ]:
# Festival vs non-festival by category
fest_comp = df.groupby(['Category','is_festival_period'])['TotalVolume'].mean().unstack()
fest_comp.columns = ['Non-Festival','Festival']
fest_comp.plot(kind='bar', figsize=(12,5), color=['#3498db','#e74c3c'],
               title='Average Volume: Festival vs Non-Festival by Category')
plt.xticks(rotation=45)
plt.ylabel('Average Volume (₹)')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
from feature_engineering import engineer_features
df = engineer_features(df)

# Verify cyclical encoding
print('Month range:', df['Month'].min(), '-', df['Month'].max())
print('Month_sin range:', df['Month_sin'].min():.3f, '-', df['Month_sin'].max():.3f)
print('Month_cos range:', df['Month_cos'].min():.3f, '-', df['Month_cos'].max():.3f)

# Visualise cyclical encoding
months = np.arange(1, 13)
sin_vals = np.sin(2 * np.pi * months / 12)
cos_vals = np.cos(2 * np.pi * months / 12)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(months, sin_vals, 'bo-', label='sin(month)')
axes[0].plot(months, cos_vals, 'rs-', label='cos(month)')
axes[0].set_xticks(months)
axes[0].set_xlabel('Month')
axes[0].set_title('Cyclical Encoding of Month\n(Dec→Jan continuity preserved)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(sin_vals, cos_vals, c=months, cmap='hsv', s=100)
for i, m in enumerate(months):
    axes[1].annotate(str(m), (sin_vals[i], cos_vals[i]), textcoords='offset points', xytext=(5,5))
axes[1].set_xlabel('sin(month)')
axes[1].set_ylabel('cos(month)')
axes[1].set_title('Monthly Unit Circle — Equal Spacing')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. NLP Risk Analysis

In [ ]:
from nlp_analysis import compute_nlp_risk_score, add_nlp_scores

# Test a few descriptions
test_cases = [
    ('Online grocery store', 'Grocery'),
    ('Cryptocurrency exchange platform', 'Crypto'),
    ('Bitcoin gambling and casino', 'Crypto'),
    ('RBI-certified payment gateway', 'Services'),
    ('Online gaming and in-app purchases', 'Gaming'),
    ('International flight bookings', 'Travel'),
]
print('Description                              NLP Score')
print('-'*55)
for desc, cat in test_cases:
    score = compute_nlp_risk_score(desc)
    print(f'{desc:<40}  {score:.3f}')

In [ ]:
df = add_nlp_scores(df)

# NLP score by category
nlp_by_cat = df.groupby('Category')['nlp_risk_score'].mean().sort_values(ascending=False)
nlp_by_cat.plot(kind='bar', color='#e74c3c', figsize=(10,4),
                title='Average NLP Risk Score by Category')
plt.ylabel('NLP Risk Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Anomaly Detection

In [ ]:
from anomaly_detection import train_isolation_forest
os.makedirs('../models', exist_ok=True)
df = train_isolation_forest(df, model_path='../models/isolation_forest.pkl')

# Compare injected anomalies vs detected
print('Detected anomalies:', df['is_anomaly'].sum())
print('Injected anomalies:', df.get('is_anomaly_injected', pd.Series([0])).sum())

fig, ax = plt.subplots(figsize=(10,4))
ax.hist(df[df['is_anomaly']==0]['anomaly_score'], bins=50,
        alpha=0.7, label='Normal', color='#2ecc71')
ax.hist(df[df['is_anomaly']==1]['anomaly_score'], bins=30,
        alpha=0.7, label='Anomaly', color='#e74c3c')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Count')
ax.set_title('Isolation Forest Anomaly Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Clustering — Elbow & Silhouette

In [ ]:
from preprocessing import preprocess
from clustering import elbow_analysis, train_kmeans

df_scaled = preprocess(df.copy(), scaler_path='../models/scaler.pkl', fit=True)
results = elbow_analysis(df_scaled, plot_path='../data/elbow_plot.png')

print('\nSilhouette scores by k:')
for k, s in zip(range(2, 11), results['silhouettes']):
    bar = '█' * int(s * 30)
    print(f'  k={k:2d}  {bar}  {s:.4f}')

In [ ]:
df_scaled = train_kmeans(df_scaled, model_path='../models/kmeans.pkl')
df['cluster_id']    = df_scaled['cluster_id'].values
df['cluster_label'] = df_scaled['cluster_label'].values

# Show cluster profiles
profile_cols = ['TransactionCount','TotalVolume','RefundRate','ChargebackRate']
profile = df.groupby('cluster_label')[profile_cols].mean()
print('\nCluster Profiles (unscaled means):')
print(profile.round(4))

## 7. Risk Scoring

In [ ]:
from risk_scoring import full_risk_pipeline
df = full_risk_pipeline(df, train_weights=True)

print('Risk Level Distribution:')
print(df['risk_level'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk score histogram
axes[0].hist(df['risk_score'], bins=50, color='#3498db', edgecolor='white')
axes[0].axvline(0.35, color='orange', linestyle='--', linewidth=2, label='Low/Med boundary')
axes[0].axvline(0.65, color='red',    linestyle='--', linewidth=2, label='Med/High boundary')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Risk Score Distribution')
axes[0].legend()

# Risk level pie
rl = df['risk_level'].value_counts()
colors = ['#2ecc71','#f39c12','#e74c3c']
axes[1].pie(rl.values, labels=rl.index, colors=colors, autopct='%1.1f%%',
            startangle=90)
axes[1].set_title('Risk Level Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Save final dataset
df.to_csv('../data/risk_scored.csv', index=False)
print('Final dataset saved → ../data/risk_scored.csv')
print('Shape:', df.shape)
df[['MerchantID','Category','risk_score','risk_level','cluster_label','is_anomaly']].head(10)